# LC #15 — 3Sum [Medium]
**Category:** Two Pointers + Sort

### Problem
Given array `nums`, return all triplets `[nums[i], nums[j], nums[k]]` where `i != j != k` and `nums[i] + nums[j] + nums[k] == 0`. No duplicate triplets.
`[-1, 0, 1, 2, -1, -4]` → `[[-1,-1,2], [-1,0,1]]`

### 🎯 Core Insight
Fix one element with a for loop. Then solve Two Sum for the remaining two elements with two pointers. Skip duplicates carefully by leveraging the sorted structure.

### 🧠 Mental Models
1. **The Fixed Anchor:** "I pick one number. Then I tell my two friends (pointers) to find the remaining sum that offsets my piece."
2. **Hash Map Fallback:** "Fix one, hash the rest. But returning unique triplets gets incredibly messy to deduplicate and memory-intensive."

In [1]:
def run_tests(test_cases, func):
    print(f"\n{'='*40}\nTesting: {func.__name__}\n{'='*40}")
    passed = 0
    for i, (args, expected) in enumerate(test_cases):
        # For list of lists, sort the inner lists and the outer list for stable comparison
        result = func(*args)
        norm_result = sorted([sorted(triplet) for triplet in result])
        norm_expected = sorted([sorted(triplet) for triplet in expected])
        
        if norm_result == norm_expected:
            print(f"  ✅ Test {i+1} PASSED")
            passed += 1
        else:
            print(f"  ❌ Test {i+1} FAILED:\n    Expected: {norm_expected}\n         Got: {norm_result}")
    print(f"\n{passed}/{len(test_cases)} tests passed.\n{'='*40}\n")

In [2]:
# Brute Force (Conceptual)
# Time: O(n^3), Space: O(1) or O(n) for deduplication set

def threeSum_brute(nums: list[int]) -> list[list[int]]:
    nums.sort()
    triplets = set()
    n = len(nums)
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            for k in range(j + 1, n):
                if nums[i] + nums[j] + nums[k] == 0:
                    triplets.add((nums[i], nums[j], nums[k]))
    return [list(t) for t in triplets]

test_cases = [
    (([-1, 0, 1, 2, -1, -4],), [[-1, -1, 2], [-1, 0, 1]]),
    (([0, 1, 1],), []),
    (([0, 0, 0],), [[0, 0, 0]])
]

run_tests(test_cases, threeSum_brute)


Testing: threeSum_brute
  ✅ Test 1 PASSED
  ✅ Test 2 PASSED
  ✅ Test 3 PASSED

3/3 tests passed.



In [3]:
# Optimal: Two Pointers + Sort
# Time: O(n^2), Space: O(1) (in-place sort)

def threeSum(nums: list[int]) -> list[list[int]]:
    nums.sort()
    result = []

    for i in range(len(nums) - 2):
        # Skip duplicate values for the fixed element
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        if nums[i] > 0:      # sorted: if nums[i] > 0, no triplet sums to 0
            break

        left, right = i + 1, len(nums) - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            if total == 0:
                result.append([nums[i], nums[left], nums[right]])
                # Skip duplicates for left and right
                while left < right and nums[left] == nums[left + 1]:  left += 1
                while left < right and nums[right] == nums[right - 1]: right -= 1
                left += 1
                right -= 1
            elif total < 0:
                left += 1
            else:
                right -= 1

    return result

test_cases = [
    (([-1, 0, 1, 2, -1, -4],), [[-1, -1, 2], [-1, 0, 1]]),
    (([0, 1, 1],), []),
    (([0, 0, 0],), [[0, 0, 0]])
]

run_tests(test_cases, threeSum)


Testing: threeSum
  ✅ Test 1 PASSED
  ✅ Test 2 PASSED
  ✅ Test 3 PASSED

3/3 tests passed.



### 🎤 Interview Q&A
- **Q: Why sort first?**
  A: Sorting lets us use two pointers (sorted structure) AND skip duplicates easily (equal adjacent values).
- **Q: Why O(n²) and not O(n)?**
  A: The outer loop is O(n); the inner two-pointer scan is O(n). So O(n) × O(n) = O(n²). Cannot do better without hashing which increases space.
- **Q: What is the "early exit" optimization?**
  A: `if nums[i] > 0: break` — if the smallest remaining element is positive, the sum can only be positive. No valid triplet possible.
- **Q: How does the algorithm avoid returning duplicate triplets?**
  A: It skips duplicates at three points: when choosing the first fixed element `nums[i]`, and when moving the `left` and `right` pointers after finding a valid triplet by checking if the next element equals the previous element.
- **Q: What happens if all elements in the array are zeroes?**
  A: The array is sorted to `[0,0,0,...]`. The first iteration finds `0+0+0 = 0`, adds `[0,0,0]`, then skips all the remaining zeroes using the duplicate skipping logic, returning exactly one `[0,0,0]`.
- **Practice Problem Extension:** **LC #18 - 4Sum**. Once you understand picking a single anchor for 3Sum, 4Sum is identical — you just pick two anchors with nested loops, leaving O(n²) × O(n) = O(n³) total time.

### 💼 The Citi Angle
**Reconciling Tripartite Transactions in Global Settlements**
In a global settlement system, a transaction often involves tripartite reconciliation — checking client fiat, broker margin, and clearinghouse ledgers. When you need to find specific erroneous three-way transactions that exactly sum to 0 balance, scanning massive logs with a naive three-loop approach takes O(n³), rendering it computationally hopeless before the morning market opens. By sorting daily cash ledgers, selecting an anchor, and doing an O(n) sweep for the corresponding missing offsets, it reduces processing combinatorics down to O(n²). As told in an interview: *"Rather than grinding the server down with O(n³) SQL joins on millions of rows, we sorted isolated discrepancy logs in-memory and used fixed anchors with two-point sweeps to find all triplicate combinations summing to zero. This prevented scaling bottlenecks during audits."*